# 08 · Medir un modelo: la ecuación de Mincer y la focalización tipo Sisbén
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrezconz/investigacion-aumentada-ml/blob/main/notebooks/08_metricas_mincer_sisben.ipynb)

**Investigación Aumentada · Sesión 8 · Machine Learning: herramientas para trabajar con datos**

Dos preguntas de política pública, una por cada tipo de tarea supervisada:

1. **Predecir un número (regresión).** ¿Cuánto gana una persona según su educación y su experiencia? Es la **ecuación de Mincer**: el logaritmo del ingreso depende de los años de educación, de la experiencia y de la experiencia al cuadrado.
2. **Clasificar (clasificación).** ¿Este hogar es pobre o no? Es la lógica de la **focalización** de programas sociales, como el Sisbén. Aquí aparecen **accuracy, precision, recall, F1-score y AUC-ROC**.

> **Datos simulados.** Para que el notebook corra en cualquier lugar sin descargar microdatos, los datos se generan con reglas conocidas. Las cifras sirven para aprender a medir, no describen a Colombia. Con datos reales, la GEIH del DANE trae ingreso, educación y edad; la metodología oficial del Sisbén IV no se reproduce aquí.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix)
rng = np.random.default_rng(8)   # semilla: siempre los mismos datos

## Parte A · Mincer: predecir el ingreso

**Cómo se simulan los datos.** 3.000 personas. El ingreso sigue la ecuación de Mincer con un retorno de 9 % por año de educación, más un error al azar (la suerte, el sector, la región: todo lo que no medimos).

In [ ]:
n = 3000
educ = rng.integers(0, 19, n)                       # años de educación (0 a 18)
edad = rng.integers(18, 66, n)
exper = np.clip(edad - educ - 6, 0, None)           # experiencia potencial
ln_ing = 13.0 + 0.09*educ + 0.035*exper - 0.0005*exper**2 + rng.normal(0, 0.55, n)
mincer = pd.DataFrame({'educacion': educ, 'experiencia': exper, 'ln_ingreso': ln_ing})
mincer['ingreso'] = np.exp(mincer.ln_ingreso).round(-3)
mincer.describe().round(2)

**Paso 1. Separar primero.** El 25 % queda guardado como prueba. Ningún modelo lo ve hasta el final.

In [ ]:
X = mincer[['educacion', 'experiencia']].assign(experiencia2=mincer.experiencia**2)
y = mincer.ln_ingreso
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=8)

ols = LinearRegression().fit(X_tr, y_tr)
print('Retorno de un año de educación (β1):', round(ols.coef_[0], 4), '→', f'{(np.exp(ols.coef_[0])-1)*100:.1f} % más ingreso')

**Paso 2. Comparar con un modelo de ML.** Un bosque aleatorio no necesita que le digamos la forma de la ecuación. ¿Predice mejor? Lo medimos con **RMSE** (error promedio, en log de ingreso) y **R²** (qué parte de la variación explica) en los datos de prueba.

In [ ]:
rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=20, random_state=8).fit(X_tr, y_tr)
for nombre, m in [('Mincer (MCO)', ols), ('Bosque aleatorio', rf)]:
    pred = m.predict(X_te)
    print(f'{nombre:18s} RMSE = {mean_squared_error(y_te, pred)**0.5:.3f}   R² = {r2_score(y_te, pred):.3f}')
base = np.full(len(y_te), y_tr.mean())
print(f'{"Línea base (media)":18s} RMSE = {mean_squared_error(y_te, base)**0.5:.3f}   R² = {r2_score(y_te, base):.3f}')

**Lectura.** Los dos modelos le ganan a la línea base y empatan entre sí: cuando la relación real es la de Mincer, el ML no tiene nada que agregar. Con datos reales, donde hay sectores, regiones e interacciones, la comparación puede salir distinta.

**Tip.** Si tu pregunta es *explicar* (¿cuánto vale un año de educación?), el coeficiente de la regresión es la respuesta. Si es *predecir* (¿cuánto ganará esta persona?), compara modelos con datos de prueba.

**Paso 3. Bootstrap: ¿qué tanto cambiaría β1 con otra muestra?**

In [ ]:
betas = []
for _ in range(1000):
    idx = rng.integers(0, len(X_tr), len(X_tr))          # muestra con reemplazo
    betas.append(LinearRegression().fit(X_tr.iloc[idx], y_tr.iloc[idx]).coef_[0])
lo, hi = np.percentile(betas, [2.5, 97.5])
print(f'β1 = {ols.coef_[0]:.4f}   intervalo bootstrap del 95 %: [{lo:.4f}, {hi:.4f}]')
plt.hist(betas, bins=40, color='#2E5FAE'); plt.axvline(lo, color='#FDC50C'); plt.axvline(hi, color='#FDC50C')
plt.title('1.000 estimaciones bootstrap del retorno a la educación'); plt.xlabel('β1'); plt.show()

## Parte B · Focalización tipo Sisbén: clasificar hogares

**Cómo se simulan los datos.** 4.000 hogares con variables que suelen usarse para aproximar la capacidad de generar ingresos: educación del jefe de hogar, personas por cuarto, afiliación contributiva a salud, zona rural y si la vivienda tiene piso de tierra. Cerca de uno de cada cinco hogares es pobre.

In [ ]:
m = 4000
hog = pd.DataFrame({
    'educ_jefe': rng.integers(0, 19, m),
    'personas_cuarto': np.round(rng.gamma(2.2, 0.9, m), 1),
    'contributivo': rng.binomial(1, 0.5, m),
    'rural': rng.binomial(1, 0.25, m),
    'piso_tierra': rng.binomial(1, 0.08, m),
})
z = (-0.4 - 0.22*hog.educ_jefe + 0.75*hog.personas_cuarto - 1.1*hog.contributivo
     + 0.6*hog.rural + 1.4*hog.piso_tierra + rng.normal(0, 1.0, m))
hog['pobre'] = (z > np.quantile(z, 0.80)).astype(int)
print('Hogares pobres:', hog.pobre.mean().round(3))

In [ ]:
Xh = hog.drop(columns='pobre'); yh = hog.pobre
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, yh, test_size=0.25, random_state=8, stratify=yh)

modelos = {
    'Línea base: todos no pobres': DummyClassifier(strategy='most_frequent'),
    'Regresión logística': LogisticRegression(max_iter=1000),
    'Bosque aleatorio': RandomForestClassifier(n_estimators=300, min_samples_leaf=10, random_state=8),
}
filas = []
for nombre, mod in modelos.items():
    mod.fit(Xh_tr, yh_tr)
    p = mod.predict(Xh_te); s = mod.predict_proba(Xh_te)[:, 1]
    filas.append({'modelo': nombre,
                  'accuracy': accuracy_score(yh_te, p),
                  'precision': precision_score(yh_te, p, zero_division=0),
                  'recall': recall_score(yh_te, p),
                  'F1': f1_score(yh_te, p),
                  'AUC-ROC': roc_auc_score(yh_te, s)})
tabla = pd.DataFrame(filas).set_index('modelo').round(3)
tabla

**Lectura de la tabla.**

- **Accuracy (exactitud):** aciertos sobre el total. La línea base acierta en 80 % sin encontrar a un solo hogar pobre: por eso la exactitud engaña cuando una clase es minoritaria.
- **Precision (precisión):** de los hogares que el modelo marcó como pobres, cuántos lo eran. Si es baja, hay **errores de inclusión**: subsidios para quien no los necesita.
- **Recall (sensibilidad):** de los hogares pobres, cuántos encontró. Si es bajo, hay **errores de exclusión**: hogares pobres sin el programa.
- **F1-score:** media armónica de precision y recall. Solo sube si las dos suben.
- **AUC-ROC:** probabilidad de que el modelo le dé más puntaje a un hogar pobre que a uno no pobre escogidos al azar. 0,5 es azar; 1 es perfecto. No depende del punto de corte.

In [ ]:
log = modelos['Regresión logística']
cm = confusion_matrix(yh_te, log.predict(Xh_te), labels=[1, 0])
pd.DataFrame(cm, index=['Real: pobre', 'Real: no pobre'], columns=['Modelo: pobre', 'Modelo: no pobre'])

**La matriz de confusión.** Arriba a la derecha están los **falsos negativos** (exclusión); abajo a la izquierda, los **falsos positivos** (inclusión).

**El punto de corte es una decisión de política.** El modelo da un puntaje entre 0 y 1; por defecto, se marca como pobre a quien supera 0,5. Si se baja el corte, se excluye a menos hogares pobres pero entra más gente que no lo necesita.

In [ ]:
s = log.predict_proba(Xh_te)[:, 1]
cortes = []
for c in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    p = (s >= c).astype(int)
    cortes.append({'corte': c, 'precision': precision_score(yh_te, p), 'recall': recall_score(yh_te, p), 'F1': f1_score(yh_te, p)})
pd.DataFrame(cortes).set_index('corte').round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
for nombre in ['Regresión logística', 'Bosque aleatorio']:
    fpr, tpr, _ = roc_curve(yh_te, modelos[nombre].predict_proba(Xh_te)[:, 1])
    ax.plot(fpr, tpr, label=f'{nombre} (AUC = {tabla.loc[nombre, "AUC-ROC"]:.2f})')
ax.plot([0, 1], [0, 1], '--', color='grey', label='Azar (AUC = 0,5)')
ax.set_xlabel('Tasa de falsos positivos (inclusión)'); ax.set_ylabel('Recall (hogares pobres encontrados)')
ax.set_title('Curva ROC'); ax.legend(); plt.show()

**Robustez: validación cruzada.** Una sola partición puede tener suerte. Con cinco particiones vemos cuánto varía el F1.

In [ ]:
f1s = cross_val_score(LogisticRegression(max_iter=1000), Xh, yh, cv=5, scoring='f1')
print('F1 en 5 particiones:', f1s.round(3), '  promedio', f1s.mean().round(3), '± ', f1s.std().round(3))

## Para tu skill

```
## Medir
- Clasificar: precision, recall, F1 y AUC-ROC; la exactitud sola engaña.
- Digo qué error cuesta más (exclusión o inclusión) antes de escoger el corte.
- Predecir: RMSE y R² en datos de prueba, contra una línea base.
- Reporto un intervalo bootstrap, no solo un número.
```

**Pídele a tu asistente de IA:** «Con esta matriz de confusión, explícame qué significan los falsos negativos en mi problema, calcula precision, recall y F1, y dime qué corte escogerías si excluir a un hogar pobre cuesta el doble que incluir a uno que no lo es.»